# P02：金融数据获取、管理与初步分析

**作业编号**：ex_P02  
**姓名**：[请填入你的姓名]  
**版本号**：v1  

> 本 Notebook 为第一部分：数据下载。使用 `baostock` 和 `akshare` 获取个股行情、市场指数、宏观经济指标和财务数据。  
> AI 工具提示词：[请填入你的提示词或链接]

## 0. 环境准备与目录创建

In [2]:
# 安装依赖（首次运行时取消注释）
# !pip install baostock akshare pandas pyarrow statsmodels matplotlib seaborn scipy -q

In [3]:
import os
import time
import datetime
import pandas as pd
import baostock as bs

# ========== 项目根目录（请根据自己的路径修改）==========
PROJECT_ROOT = "dshw-p01"

# ========== 自动创建目录结构 ==========
dirs = [
    f"{PROJECT_ROOT}/data/stock",
    f"{PROJECT_ROOT}/data/index",
    f"{PROJECT_ROOT}/data/macro",
    f"{PROJECT_ROOT}/data/finance",
    f"{PROJECT_ROOT}/data/clean",
    f"{PROJECT_ROOT}/data/combined",
    f"{PROJECT_ROOT}/output",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"[OK] {d}")
print("\n目录结构创建完成！")

[OK] dshw-p01/data/stock
[OK] dshw-p01/data/index
[OK] dshw-p01/data/macro
[OK] dshw-p01/data/finance
[OK] dshw-p01/data/clean
[OK] dshw-p01/data/combined
[OK] dshw-p01/output

目录结构创建完成！


In [4]:
# ========== 创建 .gitignore ==========
with open(f"{PROJECT_ROOT}/.gitignore", "w", encoding="utf-8") as f:
    f.write("""data/stock/\ndata/index/\ndata/macro/\ndata/finance/\n*.db\n.ipynb_checkpoints/\n.DS_Store\n__pycache__/\n""")

# ========== 创建 requirements.txt ==========
with open(f"{PROJECT_ROOT}/requirements.txt", "w", encoding="utf-8") as f:
    f.write("baostock\nakshare\npandas\nnumpy\nmatplotlib\nseaborn\nstatsmodels\npyarrow\nscipy\n")

print("[OK] .gitignore 和 requirements.txt 已创建")

[OK] .gitignore 和 requirements.txt 已创建


## 1. 公共配置

In [5]:
# ========== 股票列表 ==========
STOCK_LIST = [
    {"code": "600036", "bs_code": "sh.600036", "name": "招商银行", "industry": "银行"},
    {"code": "601398", "bs_code": "sh.601398", "name": "工商银行", "industry": "银行"},
    {"code": "002594", "bs_code": "sz.002594", "name": "比亚迪",   "industry": "汽车"},
    {"code": "600519", "bs_code": "sh.600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "000858", "bs_code": "sz.000858", "name": "五粮液",   "industry": "白酒"},
    {"code": "000002", "bs_code": "sz.000002", "name": "万科A",    "industry": "房地产"},
    {"code": "601857", "bs_code": "sh.601857", "name": "中国石油", "industry": "能源"},
    {"code": "600028", "bs_code": "sh.600028", "name": "中国石化", "industry": "能源"},
    {"code": "000063", "bs_code": "sz.000063", "name": "中兴通讯", "industry": "通讯"},
    {"code": "002352", "bs_code": "sz.002352", "name": "顺丰控股", "industry": "物流"},
]

INDEX_LIST = [
    {"code": "000300", "bs_code": "sh.000300", "name": "沪深300"},
    {"code": "000905", "bs_code": "sh.000905", "name": "中证500"},
]

START_DATE = "2020-01-01"
END_DATE   = datetime.date.today().strftime("%Y-%m-%d")

print(f"数据时间范围：{START_DATE} ~ {END_DATE}")
pd.DataFrame(STOCK_LIST)[["code", "name", "industry"]]

数据时间范围：2020-01-01 ~ 2026-04-08


,code,name,industry
0,600036,招商银行,银行
1,601398,工商银行,银行
2,002594,比亚迪,汽车
3,600519,贵州茅台,白酒
4,000858,五粮液,白酒
5,000002,万科A,房地产
6,601857,中国石油,能源
7,600028,中国石化,能源
8,000063,中兴通讯,通讯
9,002352,顺丰控股,物流


In [6]:
# ========== 下载日志 ==========
LOG_FILE = f"{PROJECT_ROOT}/download_log.txt"

def write_log(status, name, info=""):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {status:<8} {name:<20} {info}\n"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line)
    print(line.strip())

# 清空旧日志
open(LOG_FILE, "w").close()
print(f"日志文件已初始化：{LOG_FILE}")

日志文件已初始化：dshw-p01/download_log.txt


In [7]:
# ========== 登录 baostock ==========
lg = bs.login()
print(f"登录状态：{lg.error_msg}")

login success!
登录状态：success


## 1.1 下载个股日度行情

使用 `baostock` 的 `bs.query_history_k_data_plus()` 接口，获取 10 只股票 2020-01-01 至今的**后复权**日度行情。  
字段：日期、开盘价、收盘价、最高价、最低价、成交量、成交额。  
`adjustflag="1"` 表示后复权。

In [8]:
def download_stock_bs(bs_code, code, name, start=START_DATE, end=END_DATE):
    """使用 baostock 下载单只股票的后复权日度行情。"""
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,high,low,close,volume,amount",
            start_date=start, end_date=end,
            frequency="d", adjustflag="1"  # 后复权
        )
        rows = []
        while (rs.error_code == '0') & rs.next():
            rows.append(rs.get_row_data())

        if not rows:
            write_log("FAILED", f"stock_{code}", "Error: No data returned")
            return None

        df = pd.DataFrame(rows, columns=rs.fields)
        for c in ["open","high","low","close","volume","amount"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["code"] = code
        df["name"] = name

        df.to_csv(f"{PROJECT_ROOT}/data/stock/stock_{code}.csv",
                  index=False, encoding="utf-8-sig")
        write_log("SUCCESS", f"stock_{code}", f"shape={df.shape}")
        return df
    except Exception as e:
        write_log("FAILED", f"stock_{code}", f"Error: {e}")
        return None

In [9]:
# ========== 批量下载 10 只股票 ==========
stock_data = {}
for s in STOCK_LIST:
    print(f"正在下载：{s['code']} {s['name']} ...")
    df = download_stock_bs(s["bs_code"], s["code"], s["name"])
    if df is not None:
        stock_data[s["code"]] = df
    time.sleep(0.5)

print(f"\n下载完成，成功 {len(stock_data)}/{len(STOCK_LIST)} 只。")

正在下载：600036 招商银行 ...
[2026-04-08 09:08:31] SUCCESS  stock_600036         shape=(1515, 9)
正在下载：601398 工商银行 ...
[2026-04-08 09:08:35] SUCCESS  stock_601398         shape=(1515, 9)
正在下载：002594 比亚迪 ...
[2026-04-08 09:08:40] SUCCESS  stock_002594         shape=(1515, 9)
正在下载：600519 贵州茅台 ...
[2026-04-08 09:08:45] SUCCESS  stock_600519         shape=(1515, 9)
正在下载：000858 五粮液 ...
[2026-04-08 09:08:48] SUCCESS  stock_000858         shape=(1515, 9)
正在下载：000002 万科A ...
[2026-04-08 09:08:56] SUCCESS  stock_000002         shape=(1515, 9)
正在下载：601857 中国石油 ...
[2026-04-08 09:09:00] SUCCESS  stock_601857         shape=(1515, 9)
正在下载：600028 中国石化 ...
[2026-04-08 09:09:06] SUCCESS  stock_600028         shape=(1515, 9)
正在下载：000063 中兴通讯 ...
[2026-04-08 09:09:10] SUCCESS  stock_000063         shape=(1515, 9)
正在下载：002352 顺丰控股 ...
[2026-04-08 09:09:18] SUCCESS  stock_002352         shape=(1515, 9)

下载完成，成功 10/10 只。


In [10]:
# 快速预览
if stock_data:
    k = list(stock_data.keys())[0]
    print(f"预览 stock_{k}:")
    display(stock_data[k].head())
    print(f"\n数据类型：\n{stock_data[k].dtypes}")

预览 stock_600036:


,date,open,high,low,close,volume,amount,code,name
0,2020-01-02,160.914095,165.526148,160.871783,164.510650,82624452,3.191181e+09,600036,招商银行
1,2020-01-03,164.806837,167.641768,164.679900,166.710895,52010611,2.045230e+09,600036,招商银行
2,2020-01-06,165.822335,169.926639,164.595275,166.033897,54887720,2.171576e+09,600036,招商银行
3,2020-01-07,167.345582,168.995766,165.441523,165.653085,34767128,1.369305e+09,600036,招商银行
4,2020-01-08,164.806837,164.806837,162.521967,162.521967,36323418,1.403861e+09,600036,招商银行



数据类型：
date       object
open      float64
high      float64
low       float64
close     float64
volume      int64
amount    float64
code       object
name       object
dtype: object


## 1.2 下载市场指数

- **沪深 300**（000300）：大盘蓝筹代表，CAPM 分析的市场基准（必选）。  
- **中证 500**（000905）：中小盘代表，与沪深 300 互补，可观察大小盘风格轮动。

In [11]:
def download_index_bs(bs_code, code, name, start=START_DATE, end=END_DATE):
    """使用 baostock 下载指数日度数据。"""
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,high,low,close,volume,amount",
            start_date=start, end_date=end,
            frequency="d"
        )
        rows = []
        while (rs.error_code == '0') & rs.next():
            rows.append(rs.get_row_data())

        if not rows:
            write_log("FAILED", f"index_{code}", "Error: No data returned")
            return None

        df = pd.DataFrame(rows, columns=rs.fields)
        for c in ["open","high","low","close","volume","amount"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df.to_csv(f"{PROJECT_ROOT}/data/index/index_{code}.csv",
                  index=False, encoding="utf-8-sig")
        write_log("SUCCESS", f"index_{code}", f"shape={df.shape}")
        return df
    except Exception as e:
        write_log("FAILED", f"index_{code}", f"Error: {e}")
        return None

In [12]:
index_data = {}
for idx in INDEX_LIST:
    print(f"正在下载指数：{idx['code']} {idx['name']} ...")
    df = download_index_bs(idx["bs_code"], idx["code"], idx["name"])
    if df is not None:
        index_data[idx["code"]] = df
    time.sleep(0.5)

print(f"\n指数下载完成，成功 {len(index_data)}/{len(INDEX_LIST)} 个。")

正在下载指数：000300 沪深300 ...
[2026-04-08 09:09:25] SUCCESS  index_000300         shape=(1515, 7)
正在下载指数：000905 中证500 ...
[2026-04-08 09:09:31] SUCCESS  index_000905         shape=(1515, 7)

指数下载完成，成功 2/2 个。


In [13]:
if "000300" in index_data:
    print("沪深300 预览：")
    display(index_data["000300"].head())
    print(f"数据量：{index_data['000300'].shape}")

沪深300 预览：


,date,open,high,low,close,volume,amount
0,2020-01-02,4121.348,4172.655,4121.348,4152.240,18211677184,2.701055e+11
1,2020-01-03,4161.218,4164.298,4131.864,4144.964,14282624512,2.152163e+11
2,2020-01-06,4120.521,4170.638,4102.379,4129.295,17530995200,2.501821e+11
3,2020-01-07,4137.401,4161.250,4135.097,4160.227,13948903168,1.963891e+11
4,2020-01-08,4139.631,4149.813,4101.980,4112.317,16758585088,2.124063e+11


数据量：(1515, 7)


## 1.3 下载宏观经济指标

baostock 不提供宏观数据接口，此处尝试用 `akshare`。  
若 akshare 因网络问题也失败，代码会提供手动下载指引（从国家统计局下载 CSV）。

1. **CPI 同比增速**（必选）：反映物价变化，与货币政策和估值密切相关。
2. **M2 同比增速**（自选）：M2 增速上升→流动性宽松→利好股市。

In [17]:
import akshare as ak

In [19]:
macro_ok = {"cpi": False, "m2": False}

In [20]:
# ========== CPI ==========
try:
    df_cpi = ak.macro_china_cpi_yearly()
    print(f"CPI 原始列名：{df_cpi.columns.tolist()}")
    display(df_cpi.head())
    
    df_cpi = df_cpi[["日期", "今值"]].copy()
    df_cpi.columns = ["date", "cpi"]
    df_cpi["date"] = pd.to_datetime(df_cpi["date"])
    df_cpi["cpi"] = pd.to_numeric(df_cpi["cpi"], errors="coerce")
    df_cpi = df_cpi[df_cpi["date"] >= "2020-01-01"].sort_values("date").reset_index(drop=True)
    df_cpi.to_csv(f"{PROJECT_ROOT}/data/macro/macro_cpi.csv",
                  index=False, encoding="utf-8-sig")
    write_log("SUCCESS", "macro_cpi", f"shape={df_cpi.shape}")
    macro_ok["cpi"] = True
    display(df_cpi.head())
except Exception as e:
    print(f"CPI 下载失败：{e}")
    write_log("FAILED", "macro_cpi", f"Error: {e}")

CPI 原始列名：['商品', '日期', '今值', '预测值', '前值']


,商品,日期,今值,预测值,前值
0,中国CPI年率报告,1986-02-01,7.1,NaN,NaN
1,中国CPI年率报告,1986-03-01,7.1,NaN,7.1
2,中国CPI年率报告,1986-04-01,7.1,NaN,7.1
3,中国CPI年率报告,1986-05-01,7.1,NaN,7.1
4,中国CPI年率报告,1986-06-01,7.1,NaN,7.1


[2026-04-08 09:12:08] SUCCESS  macro_cpi            shape=(70, 2)


,date,cpi
0,2020-01-09,4.5
1,2020-02-10,5.4
2,2020-03-10,5.2
3,2020-04-10,4.3
4,2020-05-12,3.3


In [21]:
# ========== M2 ==========
try:
    df_m2_raw = ak.macro_china_money_supply()
    print(f"M2 原始列名：{df_m2_raw.columns.tolist()}")
    display(df_m2_raw.head())
    
    cols = df_m2_raw.columns.tolist()
    df_m2 = df_m2_raw[[cols[0], cols[2]]].copy()
    df_m2.columns = ["date", "m2_yoy"]
    
    # 处理 "2026年02月份" 这种格式
    df_m2["date"] = df_m2["date"].str.replace("月份", "月").str.replace("年", "-").str.replace("月", "-01")
    df_m2["date"] = pd.to_datetime(df_m2["date"])
    df_m2["m2_yoy"] = pd.to_numeric(df_m2["m2_yoy"], errors="coerce")
    df_m2 = df_m2[df_m2["date"] >= "2020-01-01"].sort_values("date").reset_index(drop=True)
    
    df_m2.to_csv(f"{PROJECT_ROOT}/data/macro/macro_m2.csv",
                 index=False, encoding="utf-8-sig")
    write_log("SUCCESS", "macro_m2", f"shape={df_m2.shape}")
    macro_ok["m2"] = True
    display(df_m2.head())
except Exception as e:
    print(f"M2 下载失败：{e}")
    write_log("FAILED", "macro_m2", f"Error: {e}")

M2 原始列名：['月份', '货币和准货币(M2)-数量(亿元)', '货币和准货币(M2)-同比增长', '货币和准货币(M2)-环比增长', '货币(M1)-数量(亿元)', '货币(M1)-同比增长', '货币(M1)-环比增长', '流通中的现金(M0)-数量(亿元)', '流通中的现金(M0)-同比增长', '流通中的现金(M0)-环比增长']


,月份,货币和准货币(M2)-数量(亿元),货币和准货币(M2)-同比增长,货币和准货币(M2)-环比增长,货币(M1)-数量(亿元),货币(M1)-同比增长,货币(M1)-环比增长,流通中的现金(M0)-数量(亿元),流通中的现金(M0)-同比增长,流通中的现金(M0)-环比增长
0,2026年02月份,3492159.91,9.0,0.584687,1159258.82,5.9,-1.731121,151436.41,14.1,3.625196
1,2026年01月份,3471860.39,9.0,2.025077,1179680.52,4.9,2.123888,146138.60,2.7,3.452628
2,2025年12月份,3402948.06,8.5,0.980968,1155146.50,3.8,2.327986,141261.37,10.2,2.833230
3,2025年11月份,3369890.52,8.0,0.554356,1128866.64,4.9,0.795018,137369.38,10.6,1.395974
4,2025年10月份,3351312.31,8.2,-0.073312,1119962.73,6.2,-1.015713,135478.14,10.6,-0.246824


[2026-04-08 09:12:17] SUCCESS  macro_m2             shape=(74, 2)


,date,m2_yoy
0,2020-01-01,8.4
1,2020-02-01,8.8
2,2020-03-01,10.1
3,2020-04-01,11.1
4,2020-05-01,11.1


In [22]:
# ========== 备选方案：如果 akshare 也失败 ==========
if not macro_ok["cpi"] or not macro_ok["m2"]:
    print("="*60)
    print("宏观数据自动下载失败，请手动下载：")
    print()
    if not macro_ok["cpi"]:
        print("【CPI】")
        print("  来源：国家统计局 https://data.stats.gov.cn/")
        print("  路径：月度数据 → 居民消费价格指数 → 全国 → 当月同比")
        print("  整理为两列 date,cpi 保存到 data/macro/macro_cpi.csv")
        print()
    if not macro_ok["m2"]:
        print("【M2】")
        print("  来源：国家统计局 或 中国人民银行")
        print("  整理为两列 date,m2_yoy 保存到 data/macro/macro_m2.csv")
    print("="*60)
else:
    print("宏观数据全部下载成功！")

宏观数据全部下载成功！


## 1.4 下载财务指标

使用 `baostock` 的 `bs.query_profit_data()` 获取盈利能力指标：

1. **ROE（净资产收益率）**：`roeAvg` 字段  
2. **净利润率**：`npMargin` 字段

整理为长格式（code, year, indicator, value），每行为一只股票一个年度的一个指标观测。

In [23]:
# 先测试一只，查看返回字段
rs_test = bs.query_profit_data(code="sh.600036", year=2023, quarter=4)
print(f"字段列表：{rs_test.fields}")

rows = []
while (rs_test.error_code == '0') & rs_test.next():
    rows.append(rs_test.get_row_data())
if rows:
    display(pd.DataFrame(rows, columns=rs_test.fields))
else:
    print("无数据，请检查参数")

字段列表：['code', 'pubDate', 'statDate', 'roeAvg', 'npMargin', 'gpMargin', 'netProfit', 'epsTTM', 'MBRevenue', 'totalShare', 'liqaShare']


,code,pubDate,statDate,roeAvg,npMargin,gpMargin,netProfit,epsTTM,MBRevenue,totalShare,liqaShare
0,sh.600036,2024-03-26,2023-12-31,0.145016,0.436438,,148006000000.000000,5.812962,339123000000.000000,25219845601.00,20628944429.00


In [24]:
# ========== 批量下载财务数据 ==========
finance_records = []
target_years = [2020, 2021, 2022, 2023, 2024]

for s in STOCK_LIST:
    bs_code, code, name = s["bs_code"], s["code"], s["name"]
    print(f"正在下载财务数据：{code} {name} ... ", end="")
    year_ok = []

    for year in target_years:
        try:
            rs = bs.query_profit_data(code=bs_code, year=year, quarter=4)
            rows = []
            while (rs.error_code == '0') & rs.next():
                rows.append(rs.get_row_data())

            if rows:
                df_tmp = pd.DataFrame(rows, columns=rs.fields)
                row = df_tmp.iloc[-1]

                roe = pd.to_numeric(row.get("roeAvg", None), errors="coerce")
                npm = pd.to_numeric(row.get("npMargin", None), errors="coerce")

                finance_records.append({"code": code, "name": name,
                                        "year": year, "indicator": "ROE", "value": roe})
                finance_records.append({"code": code, "name": name,
                                        "year": year, "indicator": "net_profit_margin", "value": npm})
                year_ok.append(year)
        except:
            pass

    write_log("SUCCESS", f"finance_{code}", f"years={year_ok}")
    print(f"完成 {len(year_ok)} 年")
    time.sleep(0.3)

print(f"\n财务数据共 {len(finance_records)} 条记录。")

正在下载财务数据：600036 招商银行 ... [2026-04-08 09:13:11] SUCCESS  finance_600036       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：601398 工商银行 ... [2026-04-08 09:13:13] SUCCESS  finance_601398       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：002594 比亚迪 ... [2026-04-08 09:13:14] SUCCESS  finance_002594       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：600519 贵州茅台 ... [2026-04-08 09:13:18] SUCCESS  finance_600519       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：000858 五粮液 ... [2026-04-08 09:13:22] SUCCESS  finance_000858       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：000002 万科A ... [2026-04-08 09:13:30] SUCCESS  finance_000002       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：601857 中国石油 ... [2026-04-08 09:13:34] SUCCESS  finance_601857       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：600028 中国石化 ... [2026-04-08 09:13:39] SUCCESS  finance_600028       years=[2020, 2021, 2022, 2023, 2024]
完成 5 年
正在下载财务数据：000063 中兴通讯 ... [2026-04-0

In [25]:
# ========== 保存财务数据 ==========
df_finance = pd.DataFrame(finance_records)
df_finance.to_csv(f"{PROJECT_ROOT}/data/finance/finance_ratios.csv",
                  index=False, encoding="utf-8-sig")

print(f"已保存，形状：{df_finance.shape}")
print(f"指标类型：{df_finance['indicator'].unique()}")
display(df_finance.head(10))

已保存，形状：(100, 5)
指标类型：['ROE' 'net_profit_margin']


,code,name,year,indicator,value
0,600036,招商银行,2020,ROE,0.145825
1,600036,招商银行,2020,net_profit_margin,0.337229
2,600036,招商银行,2021,ROE,0.151561
3,600036,招商银行,2021,net_profit_margin,0.364779
4,600036,招商银行,2022,ROE,0.152986
5,600036,招商银行,2022,net_profit_margin,0.404005
6,600036,招商银行,2023,ROE,0.145016
7,600036,招商银行,2023,net_profit_margin,0.436438
8,600036,招商银行,2024,ROE,0.128902
9,600036,招商银行,2024,net_profit_margin,0.443154


## 1.5 下载日志 & 数据完整性检查

In [26]:
print("="*70)
print("完整下载日志 (download_log.txt)")
print("="*70)
with open(LOG_FILE, "r", encoding="utf-8") as f:
    print(f.read())

完整下载日志 (download_log.txt)
[2026-04-08 09:08:31] SUCCESS  stock_600036         shape=(1515, 9)
[2026-04-08 09:08:35] SUCCESS  stock_601398         shape=(1515, 9)
[2026-04-08 09:08:40] SUCCESS  stock_002594         shape=(1515, 9)
[2026-04-08 09:08:45] SUCCESS  stock_600519         shape=(1515, 9)
[2026-04-08 09:08:48] SUCCESS  stock_000858         shape=(1515, 9)
[2026-04-08 09:08:56] SUCCESS  stock_000002         shape=(1515, 9)
[2026-04-08 09:09:00] SUCCESS  stock_601857         shape=(1515, 9)
[2026-04-08 09:09:06] SUCCESS  stock_600028         shape=(1515, 9)
[2026-04-08 09:09:10] SUCCESS  stock_000063         shape=(1515, 9)
[2026-04-08 09:09:18] SUCCESS  stock_002352         shape=(1515, 9)
[2026-04-08 09:09:25] SUCCESS  index_000300         shape=(1515, 7)
[2026-04-08 09:09:31] SUCCESS  index_000905         shape=(1515, 7)
[2026-04-08 09:09:31] FAILED   macro_cpi            Error: name 'ak' is not defined
[2026-04-08 09:09:31] FAILED   macro_m2             Error: name 'ak' is no

In [27]:
# ========== 个股数据汇总表 ==========
summary = []
for s in STOCK_LIST:
    fpath = f"{PROJECT_ROOT}/data/stock/stock_{s['code']}.csv"
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        summary.append({"code": s["code"], "name": s["name"], "industry": s["industry"],
                        "rows": len(df), "date_min": df["date"].min(),
                        "date_max": df["date"].max(), "missing": df.isnull().sum().sum()})
    else:
        summary.append({"code": s["code"], "name": s["name"], "industry": s["industry"],
                        "rows": 0, "date_min": "-", "date_max": "-", "missing": "-"})

print("个股数据汇总：")
display(pd.DataFrame(summary))

个股数据汇总：


,code,name,industry,rows,date_min,date_max,missing
0,600036,招商银行,银行,1515,2020-01-02,2026-04-07,0
1,601398,工商银行,银行,1515,2020-01-02,2026-04-07,0
2,002594,比亚迪,汽车,1515,2020-01-02,2026-04-07,0
3,600519,贵州茅台,白酒,1515,2020-01-02,2026-04-07,0
4,000858,五粮液,白酒,1515,2020-01-02,2026-04-07,0
5,000002,万科A,房地产,1515,2020-01-02,2026-04-07,0
6,601857,中国石油,能源,1515,2020-01-02,2026-04-07,0
7,600028,中国石化,能源,1515,2020-01-02,2026-04-07,0
8,000063,中兴通讯,通讯,1515,2020-01-02,2026-04-07,0
9,002352,顺丰控股,物流,1515,2020-01-02,2026-04-07,0


In [28]:
print("--- 指数数据 ---")
for idx in INDEX_LIST:
    fp = f"{PROJECT_ROOT}/data/index/index_{idx['code']}.csv"
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        print(f"  {idx['name']}({idx['code']}): {len(df)} 行, {df['date'].min()} ~ {df['date'].max()}")

print("\n--- 宏观数据 ---")
for fn in ["macro_cpi.csv", "macro_m2.csv"]:
    fp = f"{PROJECT_ROOT}/data/macro/{fn}"
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        print(f"  {fn}: {len(df)} 行, {df['date'].min()} ~ {df['date'].max()}")
    else:
        print(f"  {fn}: 未找到（需手动下载）")

print("\n--- 财务数据 ---")
fp = f"{PROJECT_ROOT}/data/finance/finance_ratios.csv"
if os.path.exists(fp):
    df = pd.read_csv(fp)
    print(f"  共 {len(df)} 条, {df['code'].nunique()} 只股票, 年份 {sorted(df['year'].unique())}")

--- 指数数据 ---
  沪深300(000300): 1515 行, 2020-01-02 ~ 2026-04-07
  中证500(000905): 1515 行, 2020-01-02 ~ 2026-04-07

--- 宏观数据 ---
  macro_cpi.csv: 70 行, 2020-01-09 ~ 2025-09-10
  macro_m2.csv: 74 行, 2020-01-01 ~ 2026-02-01

--- 财务数据 ---
  共 100 条, 10 只股票, 年份 [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [29]:
# ========== 登出 baostock ==========
bs.logout()
print("baostock 已登出。")

logout success!
baostock 已登出。


## 小结

| 数据类型 | 数量 | 来源 | 存储位置 |
|---------|------|------|----------|
| 个股日度行情（后复权） | 10 只 | baostock `query_history_k_data_plus()` | `data/stock/` |
| 市场指数 | 沪深300 + 中证500 | baostock | `data/index/` |
| CPI 同比、M2 同比 | 月度 | akshare（或手动） | `data/macro/` |
| ROE、净利润率 | 10只×5年 | baostock `query_profit_data()` | `data/finance/` |

所有记录已写入 `download_log.txt`。**下一步**：运行 `02_clean.ipynb` 进行数据清洗。